# Maxwell Modernized -- Electromagnetism Deep Dive

A comprehensive exploration of electromagnetic phenomena from Maxwell's Treatise:

1. Electromagnetic field setup and Maxwell equations
2. Poynting vector computation
3. Maxwell stress tensor
4. Faraday induction analysis
5. Ampere-Maxwell law and displacement current
6. Electromagnetic energy

Based on Part IV of Maxwell's *A Treatise on Electricity and Magnetism* (1873).

## 1. Electromagnetic Field Setup

The unified electromagnetic field theory (Arts. 475-866).

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from maxwell import MaxwellEquations, ElectromagneticField
from maxwell import C, CONST

print(f"Maxwell equations implementation loaded")
print(f"Speed of light: c = {C:.4e} cm/s")

## 2. Point Charge and Field Visualization

Exploring the electric field of a point charge in 2D.

In [ ]:
from maxwell import PointCharge

# Create a point charge at the origin
q = PointCharge(q=1.0, position=np.array([0.0, 0.0, 0.0]))

# Evaluate field on a 2D grid (z=0 plane)
x = np.linspace(-3, 3, 30)
y = np.linspace(-3, 3, 30)
X, Y = np.meshgrid(x, y)

# Compute Ex and Ey at each grid point
Ex = np.zeros_like(X)
Ey = np.zeros_like(Y)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        point = np.array([X[i, j], Y[i, j], 0.0])
        E = q.field_at(point)
        Ex[i, j] = E[0]
        Ey[i, j] = E[1]

fig, ax = plt.subplots(figsize=(10, 8))
magnitude = np.sqrt(Ex**2 + Ey**2)
ax.streamplot(X, Y, Ex, Ey, density=1.5, color=magnitude, cmap='autumn')
ax.plot(0, 0, 'ro', markersize=12, label='+q')
ax.set_xlabel('x (cm)')
ax.set_ylabel('y (cm)')
ax.set_title('Electric Field Lines of a Point Charge (Art. 29-30)')
ax.legend()
ax.set_aspect('equal')
plt.show()

## 3. Electric Dipole

Superposition of two opposite charges.

In [ ]:
q_plus = PointCharge(q=1.0, position=np.array([1.0, 0.0, 0.0]))
q_minus = PointCharge(q=-1.0, position=np.array([-1.0, 0.0, 0.0]))

# Compute combined field
Ex = np.zeros_like(X)
Ey = np.zeros_like(Y)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        point = np.array([X[i, j], Y[i, j], 0.0])
        E = q_plus.field_at(point) + q_minus.field_at(point)
        Ex[i, j] = E[0]
        Ey[i, j] = E[1]

fig, ax = plt.subplots(figsize=(10, 8))
magnitude = np.sqrt(Ex**2 + Ey**2)
ax.streamplot(X, Y, Ex, Ey, density=1.5, color=magnitude, cmap='coolwarm')
ax.plot(1, 0, 'ro', markersize=12, label='+q')
ax.plot(-1, 0, 'bo', markersize=12, markerfacecolor='none', markeredgewidth=2, label='-q')
ax.set_xlabel('x (cm)')
ax.set_ylabel('y (cm)')
ax.set_title('Electric Field of a Dipole')
ax.legend()
ax.set_aspect('equal')
plt.show()

## 4. Lorentz Force Analysis

Complete force analysis for a current-carrying wire in a magnetic field (Art. 490-492).

In [ ]:
from maxwell.electromagnetism.forces.lorentz import analyze_lorentz_force

# Wire: 1 abampere, 10 cm along x-axis
# Field: 1000 gauss in z-direction
result = analyze_lorentz_force(
    current=1.0,
    wire_length=np.array([10.0, 0.0, 0.0]),
    B_field=np.array([0.0, 0.0, 1000.0]),
    wire_mass=0.1,  # 0.1 gram
)

print(f"Force vector: {result['force_vector']} dynes")
print(f"Force magnitude: {result['force_magnitude']:.2f} dynes")
print(f"Angle between wire and field: {np.degrees(result['angle_wire_field']):.1f} degrees")
print(f"Efficiency (sin theta): {result['efficiency']:.4f}")
print(f"Acceleration: {result['acceleration']:.2f} cm/s^2")

## 5. Parallel Currents Force (Art. 492)

Two parallel wires carrying currents exert forces on each other.

In [ ]:
from maxwell.electromagnetism.forces.lorentz import (
    calc_force_between_parallel_currents,
    verify_parallel_current_attraction,
)

# Two 1 abampere wires, 1 cm apart, 10 cm length
F = calc_force_between_parallel_currents(1.0, 1.0, 1.0, 10.0)
print(f"Force between parallel currents: F = {F:.2f} dynes (attractive)")

# Verify the attraction law
result = verify_parallel_current_attraction()
print(f"Attraction verified: {result['attraction_verified']}")
print(f"Repulsion verified: {result['repulsion_verified']}")
print(f"Inverse-r law verified: {result['inverse_r_verified']}")

## 6. Maxwell Stress Tensor

The stress tensor describes electromagnetic field momentum flux (Arts. 617-620).

In [ ]:
from maxwell import MaxwellStressTensor

# E field in x-direction, B field in z-direction
E = np.array([100.0, 0.0, 0.0])  # statvolt/cm
B = np.array([0.0, 0.0, 1000.0])  # gauss

stress = MaxwellStressTensor(E_field=E, B_field=B)
print(f"Stress tensor (dynes/cm^2):")
print(f"T = {stress.tensor}")
print(f"Trace: {stress.trace:.4f}")
print(f"Symmetric: {stress.is_symmetric}")

## 7. Poynting Vector

The energy flux of the electromagnetic field (Art. 620).

In [ ]:
from maxwell.electromagnetism.theory.general_equations import ElectromagneticField

# E in x, B in z -> S in y-direction (E x B)
em = ElectromagneticField(
    E_field=np.array([100.0, 0.0, 0.0]),
    B_field=np.array([0.0, 0.0, 1000.0]),
)
print(f"Poynting vector S = {em.poynting_vector}")
print(f"Energy density u = {em.energy_density:.4f} erg/cm^3")

## 8. Faraday Induction -- Complete Analysis

Comprehensive induction scenario for a multi-turn coil (Art. 528-531).

In [ ]:
from maxwell.electromagnetism.induction.faraday import analyze_faraday_induction

# 100-turn coil, field changes from 0 to 1000 gauss
result = analyze_faraday_induction(
    B_initial=np.array([0.0, 0.0, 0.0]),
    B_final=np.array([0.0, 0.0, 1000.0]),
    loop_area=10.0,       # 10 cm^2
    loop_normal=np.array([0.0, 0.0, 1.0]),
    time_interval=0.5,    # 0.5 seconds
    resistance=100.0,
    num_turns=100,
)

print(f"Initial flux per turn: {result['flux_initial']:.2f} maxwells")
print(f"Final flux per turn: {result['flux_final']:.2f} maxwells")
print(f"Total flux change: {result['total_flux_change']:.2f} maxwells")
print(f"Induced EMF: {result['average_emf']:.2f} abvolts")
print(f"Induced current: {result['average_current']:.4f} abamperes")
print(f"Charge transferred: {result['charge_transferred']:.4f} abcoulombs")

## 9. Lenz's Law Verification (Art. 542)

The induced current always opposes the change in flux.

In [ ]:
from maxwell.electromagnetism.induction.faraday import verify_lenz_law

# Increasing flux: 1000 -> 2000 maxwells over 0.1 s
result = verify_lenz_law(
    initial_flux=1000.0,
    final_flux=2000.0,
    time_interval=0.1,
    resistance=10.0,
)

print(f"Flux change: {result['flux_change']:.2f} maxwells")
print(f"Induced EMF: {result['induced_emf']:.2f} abvolts")
print(f"Induced current: {result['induced_current']:.4f} abamperes")
print(f"Lenz's law verified: {result['lenz_law_verified']}")
print(f"Opposes change: {result['opposes_change']}")

## 10. Ampere-Maxwell Law (Art. 616)

The generalized Ampere's law including displacement current.

In [ ]:
from maxwell.electromagnetism.fields.ampere_maxwell import (
    AmpereMaxwellLaw,
    DisplacementCurrent,
)

# Displacement current for a changing electric field
dE_dt = np.array([1e6, 0.0, 0.0])  # statvolt/(cm*s)
dc = DisplacementCurrent(dE_dt=dE_dt)
print(f"Displacement current density: {dc.current_density}")

# Full Ampere-Maxwell law
ampere = AmpereMaxwellLaw(
    current_density=np.array([10.0, 0.0, 0.0]),
    dE_dt=dE_dt,
)
print(f"Total curl B = {ampere.total_source}")

## 11. Magnetic Energy (Art. 635)

Energy stored in a magnetic field.

In [ ]:
from maxwell.electromagnetism.energy.magnetic import (
    calc_magnetic_energy_density,
    calc_total_magnetic_energy,
)

B = np.array([0.0, 0.0, 1000.0])  # gauss
u = calc_magnetic_energy_density(B)
print(f"Magnetic energy density: u = {u:.4f} erg/cm^3")

# Total energy in 1 cm^3 volume
U = calc_total_magnetic_energy(B, volume=1.0)
print(f"Total magnetic energy (1 cm^3): U = {U:.4f} erg")

## 12. Electrostatic Energy (Art. 36)

Energy stored in an electrostatic field.

In [ ]:
from maxwell.electromagnetism.energy.electrostatic import (
    calc_electrostatic_energy_density,
)

E = np.array([100.0, 0.0, 0.0])  # statvolt/cm
u = calc_electrostatic_energy_density(E)
print(f"Electrostatic energy density: u = {u:.4f} erg/cm^3")

## 13. Stress Tensor Visualization

Visualizing the Maxwell stress tensor field.

In [ ]:
from maxwell.vis import plot_stress_tensor_2d

fig, ax = plot_stress_tensor_2d(
    E_strength=100.0,
    B_strength=1000.0,
    x_range=(-3.0, 3.0),
    y_range=(-3.0, 3.0),
)
plt.show()

## 14. Force on a Moving Charge (Art. 491)

The magnetic force on a charge moving in a magnetic field.

In [ ]:
from maxwell.electromagnetism.forces.lorentz import (
    calc_force_on_moving_charge,
    calc_force_density,
)

# Electron (1.6e-20 abcoulombs) moving at 1e9 cm/s in 1000 gauss field
F = calc_force_on_moving_charge(
    charge=1.6e-20,
    velocity=np.array([1e9, 0.0, 0.0]),
    B_field=np.array([0.0, 0.0, 1000.0]),
)
print(f"Force on moving charge: F = {F} dynes")

# Force density (force per unit volume)
J = np.array([1.0, 0.0, 0.0])  # abA/cm^2
B = np.array([0.0, 0.0, 1000.0])  # gauss
f = calc_force_density(J, B)
print(f"Force density: f = {f} dynes/cm^3")

## 15. Field Lines Visualization

Electric field line plot using the built-in visualization module.

In [ ]:
from maxwell.vis import plot_field_lines_2d

# Create a dipole field using lambda functions
q_plus = PointCharge(q=1.0, position=np.array([1.0, 0.0, 0.0]))
q_minus = PointCharge(q=-1.0, position=np.array([-1.0, 0.0, 0.0]))

fig, ax = plt.subplots(figsize=(10, 8))

# Compute field on grid
x = np.linspace(-3, 3, 40)
y = np.linspace(-3, 3, 40)
X, Y = np.meshgrid(x, y)
Ex, Ey = np.zeros_like(X), np.zeros_like(Y)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        point = np.array([X[i, j], Y[i, j], 0.0])
        E = q_plus.field_at(point) + q_minus.field_at(point)
        Ex[i, j] = E[0]
        Ey[i, j] = E[1]

ax.streamplot(X, Y, Ex, Ey, density=2.0, color='blue', linewidth=0.8)
ax.plot(1, 0, 'ro', markersize=12, label='+q')
ax.plot(-1, 0, 'bo', markersize=12, markerfacecolor='none', markeredgewidth=2, label='-q')
ax.set_xlabel('x (cm)')
ax.set_ylabel('y (cm)')
ax.set_title('Dipole Field Lines')
ax.legend()
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.show()

## 16. Motional EMF (Art. 529-530)

EMF induced when a conductor moves through a magnetic field.

In [ ]:
from maxwell.electromagnetism.induction.faraday import calc_motional_emf

# Conductor moving at 100 cm/s perpendicular to 1000 gauss field
v = np.array([100.0, 0.0, 0.0])
B = np.array([0.0, 0.0, 1000.0])
emf = calc_motional_emf(v, B, conductor_length=10.0)
print(f"Motional EMF: {emf:.2f} abvolts")

## 17. Self-Induction (Art. 529, 542)

EMF from self-inductance opposing current changes.

In [ ]:
from maxwell.electromagnetism.induction.faraday import calc_self_induction

# 1000 cm inductance, current increasing at 10 A/s
emf = calc_self_induction(1000.0, 10.0)
print(f"Self-induced EMF: {emf:.2f} abvolts")
print(f"(Negative sign = opposes current increase)")

## 18. Summary of Electromagnetic Laws

The complete set of Maxwell's electromagnetic laws implemented in this library:

In [ ]:
laws = [
    ("Art. 29-30", "Point charge field: E = q/r^2"),
    ("Art. 72", "Electric field from potential gradient: E = -grad(V)"),
    ("Art. 490-491", "Lorentz force: F = I L x B = q v x B"),
    ("Art. 492", "Parallel currents: F/L = 2 I1 I2 / r"),
    ("Art. 529", "Faraday's law: EMF = -d(phi)/dt"),
    ("Art. 542", "Lenz's law: induced EMF opposes flux change"),
    ("Art. 616", "Ampere-Maxwell law: curl B = (4 pi/c) J + (1/c) dE/dt"),
    ("Art. 617-620", "Maxwell stress tensor and Poynting vector"),
    ("Art. 620", "Poynting vector: S = (c/4 pi) E x B"),
    ("Art. 635", "Magnetic energy density: u = B^2 / (8 pi)"),
    ("Art. 782", "Speed of light: c = 1/sqrt(mu_0 epsilon_0)"),
]

print(f"{'Article':<12} {'Law'}")
print("-" * 70)
for article, law in laws:
    print(f"{article:<12} {law}")

## Summary

This notebook covered the core electromagnetic phenomena from Part IV of Maxwell's Treatise:

1. Electric field computation for point charges and dipoles
2. Lorentz force analysis (wire, moving charge, parallel currents)
3. Maxwell stress tensor properties
4. Poynting vector and energy density
5. Faraday induction with Lenz's law
6. Ampere-Maxwell law with displacement current
7. Magnetic and electrostatic energy
8. Motional EMF and self-induction

For visualizations of specific Maxwell articles, see `03-visualization-showcase.ipynb`.